<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.1: 介绍 to FIRRTL

**Prev: [Generators: Types](3.6_types.ipynb)**<br>
**Next: [FIRRTL AST Traversal](4.2_firrtl_ast_traversal.ipynb)**

## 动机
You've learned some Scala and written some Chisel, and for 90% of users, that should be enough to become a Chisel aficionado.

然而, some use cases are better expressed as a programmatic transformation of a Chisel 设计, rather than as a generator.

例如, suppose we want to count the number of registers in a 设计. This would be difficult to do as a generator, so instead, we can write a FIRRTL pass to do it for us.

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}
import firrtl._

## What is FIRRTL?
As you've probably become aware, when you execute a Chisel 设计, it elaborates (executes the surrounding Scala code) to construct an 实例 of your generator, with all Scala parameters resolved.

Instead of directly emitting Verilog, Chisel emits an intermediate representation called FIRRTL, which represents the elaborated (参数-resolved) RTL 实例. It can be serialized (converted to a String for writing to a file), and this serialized syntax is human readable. Internally, 然而, it is not represented as a long string. Instead, it is a datastructure organized as a tree of nodes, called an abstract-syntax-tree (AST).

Let's take a look! We will take a simple Chisel 设计, elaborate it, and inspect what FIRRTL it generates!

First, we define a Chisel 模块, which delays its 输入 signal by two cycles.

In [ ]:
class DelayBy2(width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  val r0 = RegNext(io.in)
  val r1 = RegNext(r0)
  io.out := r1
}

Next, let's elaborate it, serialize, and print out the FIRRTL it generates.

In [ ]:
println(chisel3.Driver.emit(() => new DelayBy2(4)))

As you can see, the serialized FIRRTL looks very similar to what our Chisel 设计 would look like, with all generator parameters resolved.

## The FIRRTL AST

As mentioned earlier, the FIRRTL representation can be serialized as a String, but internally, it is a datastructure called an AST (abstract syntax tree). This data structure is a tree of nodes, where one node can contain children nodes. There are no cycles in this datastructure.

Let's take a look at what the internal datastructure looks like:

In [ ]:
val firrtlSerialization = chisel3.Driver.emit(() => new DelayBy2(4))
val firrtlAST = firrtl.Parser.parse(firrtlSerialization.split("\n").toIterator, Parser.GenInfo("file.fir"))

println(firrtlAST)

Obviously, the serialization of a datastructure isn't as pretty, but you can see some of the classes and such that internally represent the RTL 设计. Let's try to pretty that up a bit to make it understandable.

In [ ]:
println(stringifyAST(firrtlAST))

This is the internal datastructure that holds the FIRRTL AST. It is a tree structure whose root node is **电路**, which has 3 children: **@[file.fir@2.0]**, **ArrayBuffer**, and **cmd5WrapperHelperDelayBy2**. 以下 is the definition of `电路`'s actual Scala 类 that was serialized:<a name="电路"></a><img src="images/电路.png" alt="电路 case 类" />



As you can see, it has three children nodes: `info: Info`, `Modules: Seq[DefModule]`, and `main: String`. It extends `FirrtlNode`, of which all FIRRTL AST nodes must do. Ignore the `def mapXXXX` functions for now.

Many FIRRTL nodes contain an `info: Info` field, which the parser can either insert file information like line number and column number, or insert a `NoInfo` token. 在这个例子中, **@[file.fir@2.0]** would refer to the FIRRTL file, line 2, column 0.

以下 section will outline all of these FIRRTL nodes in detail.

# FIRRTL Node Descriptions

This section describes common FirrtlNodes found in [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala).

For more detail on components not mentioned here, please refer to [The FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf).


## 电路
电路 is the root node of any Firrtl datastructure. There is only ever one 电路, and that 电路 contains a list of 模块 definitions and the name of the top-level 模块.

#### FirrtlNode Declaration
```scala 
电路(info: Info, modules: Seq[DefModule], main: String)
```

#### Concrete Syntax
```
电路 Adder:
  ... //List of modules
```
#### In-memory Representation
```scala
电路(NoInfo, Seq(...), "Adder")
```

## 模块

Modules are the unit of modularity within Firrtl and are never directly nested (declaring an 实例 of a 模块 has its own concrete syntax and AST representation). Each 模块 has a name, and a list of ports, and a body containing its 实现.

#### FirrtlNode declaration
```scala
模块(info: Info, name: String, ports: Seq[Port], body: Stmt) extends DefModule
```

#### Concrete Syntax
```
模块 Adder:
  ... // list of ports
  ... // statements
```
#### In-memory representation
```scala
模块(NoInfo, "Adder", Seq(...), )
```

## Port
A port defines part of a 模块's io, and has a name, direction (输入 or 输出), and 类型.

#### FirrtlNode Declaration
```scala
类 Port(info: Info, name: String, direction: Direction, tpe: 类型)
```
#### Concrete Syntax
```
输入 x: UInt
```

#### In-memory representation
```scala
Port(NoInfo, "x", 输入, UIntType(UnknownWidth))
```

## Statement
A statement is used to describe the components within a 模块 and how they interact. Below are some commonly used statements:

### Block of Statements
A group of statements. Commonly used as the body field in a 模块 declaration.

### 导线 Declaration
A 导线 declaration, containing a name and 类型. It can be both a source (connected *from*) and a sink (connected *to").
#### FirrtlNode declaration
```scala
DefWire(info: Info, name: String, tpe: 类型)
```
#### Concrete syntax
```
导线 w: UInt
```
#### In-memory Representation
```scala
DefWire(NoInfo, "w", UIntType(UnknownWidth))
```

### 寄存器 Declaration
A 寄存器 declaration, containing a name, 类型, 时钟 signal, 复位 signal, and 复位 值.
#### FirrtlNode declaration
```scala
DefRegister(info: Info, name: String, tpe: 类型, 时钟: Expression, 复位: Expression, init: Expression)
```

### Connection
Represents a directioned connection from a source to a sink. 请注意 it abides by last-connect-semantics, as described in Chisel.

#### FirrtlNode declaration
```scala
Connect(info: Info, loc: Expression, expr: Expression)
```

### Other Statements
Other statement types like `DefMemory`, `DefNode`, `IsInvalid`, `Conditionally`, and others are omitted here; please refer to [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/freechipsproject/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) for more detail.

## Expression
Expressions represent 参考文献 to declared components or logical and arithmetic operations. Below are some commonly used expressions:

### Reference
A reference to a declared component, such as a 导线, 寄存器, or port. It has a name and 类型 field. 请注意 it does not contain a pointer to the actual declaration, but instead just contains the name as a String.

#### FirrtlNode declaration
```scala
Reference(name: String, tpe: 类型)
```

### DoPrim
An anonymous primitive operation, such as `Add`, `Sub`, or `And`, `Or`, or subword-selection (`Bits`). The 类型 of operation is indicated by the `op: PrimOp` field. 请注意 the number of required arguments and constants are determined by the `op`.

#### FirrtlNode declaration
```scala
DoPrim(op: PrimOp, args: Seq[Expression], consts: Seq[BigInt], tpe: 类型)
```

### Other Expressions
Other expressions including `SubField`, `SubIndex`, `SubAccess`, `Mux`, `ValidIf` etc. are described in more detail in [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) and [The FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf).

# Back to our 示例

Let's take another look at the FIRRTL AST from our 示例. Hopefully, the structure of the 设计 makes more sense!

In [ ]:
println(stringifyAST(firrtlAST))

That's it for this section! In the next section, we will look at how a FIRRTL transformation walks this AST and modifies it.